# R04. Contest Choice

In [35]:
import pandas as pd
import glob
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

payout_folder = r"C:\Users\James\Documents\MLB\Data\A09. DraftKings\3. Payouts"
entry_results_folder = r"C:\Users\James\Documents\MLB\Data\A09. DraftKings\5. Entry Results"

PAYOUT_COLS = ['contestKey', 'name', 'entryFee', 'entries', 'payoutDescription',
               'maxPosition', 'minPosition', 'maximumEntriesPerUser',
               'draftGroupId', 'contestStartTime']

def process_file(f):
    df = pd.read_csv(f, usecols=lambda c: c in PAYOUT_COLS)
    first = df.iloc[0]
    contest_key = first['contestKey']

    # --- Entry results ---
    entry_file = os.path.join(entry_results_folder, f"Entry Results {int(contest_key)}.csv")
    if os.path.exists(entry_file):
        try:
            entry_df = pd.read_csv(entry_file, encoding='iso-8859-1',
                                   usecols=['Points'], engine='pyarrow')
        except Exception:
            entry_df = pd.read_csv(entry_file, encoding='iso-8859-1', usecols=['Points'])

        actual_entries = len(entry_df)
        average_score = entry_df['Points'].mean()
        std_score     = entry_df['Points'].std()

        # --- NEW: Percentiles ---
        p95_score = entry_df['Points'].quantile(0.95)
        p99_score = entry_df['Points'].quantile(0.99)

    else:
        actual_entries = average_score = std_score = None
        p95_score = p99_score = None

    # --- Payout calc ---
    payout_clean = (
        df['payoutDescription']
            .astype(str)
            .str.replace(',', '', regex=False)
            .str.extract(r'(\$?\d+\.?\d*)')[0]
            .str.replace('$', '', regex=False)
            .astype(float)
            .fillna(0.0)
    )
    total_payout = ((df['maxPosition'] - df['minPosition'] + 1) * payout_clean).sum()

    # --- Derived metrics ---
    entry_fee = first['entryFee']
    if actual_entries:
        total_entry_fees = actual_entries * entry_fee
        ev_ratio  = total_payout / total_entry_fees
        rake_pct  = 1 - ev_ratio
    else:
        total_entry_fees = ev_ratio = rake_pct = None

    return {
        'contestKey':             contest_key,
        'contestName':            first['name'],
        'entryFee':               entry_fee,
        'configured_entries':     first['entries'],
        'actual_entries':         actual_entries,
        'average_score':          average_score,
        'std_score':              std_score,
        'p95_score':              p95_score,
        'p99_score':              p99_score,
        'maximumEntriesPerUser':  first.get('maximumEntriesPerUser'),
        'draftGroupId':           first.get('draftGroupId'),
        'contestStartTime':       first.get('contestStartTime'),
        'total_entry_fees':       total_entry_fees,
        'total_payout':           total_payout,
        'ev_ratio':               ev_ratio,
        'rake_pct':               rake_pct,
    }

files = glob.glob(os.path.join(payout_folder, "*.csv"))

results = []
with ThreadPoolExecutor() as executor:
    futures = {executor.submit(process_file, f): f for f in files}
    for future in as_completed(futures):
        try:
            results.append(future.result())
        except Exception as e:
            print(f"Error processing {futures[future]}: {e}")

final_df = pd.DataFrame(results)

In [37]:
final_df.tail()

,contestKey,contestName,entryFee,configured_entries,actual_entries,average_score,std_score,p95_score,p99_score,maximumEntriesPerUser,draftGroupId,contestStartTime,total_entry_fees,total_payout,ev_ratio,rake_pct
14459,189667120,MLB $200K Bat Flip [$50K to 1st],18.0,999,13065.0,114.181427,22.022150,150.80,163.200,150,145661,2026-04-15T23:05:00.0000000Z,235170.0,200000.0,0.850449,0.149551
14460,189667205,MLB $3K Chin Music [Single Entry] (Early),5.0,287,707.0,77.781047,23.985934,115.80,137.668,1,145660,2026-04-15T16:35:00.0000000Z,3535.0,3000.0,0.848656,0.151344
14461,189685166,MLB $6K Chin Music [Single Entry] (Night),5.0,79,1412.0,81.253471,17.313887,115.25,123.250,1,145909,2026-04-17T00:10:00.0000000Z,7060.0,6000.0,0.849858,0.150142
14462,189685173,MLB $75K Relay Throw [$20K to 1st] (Night),15.0,231,5848.0,81.856721,17.662649,114.25,125.250,150,145909,2026-04-17T00:10:00.0000000Z,87720.0,75000.0,0.854993,0.145007
14463,189685118,MLB $15K mini-MAX [150 Entry Max],1.0,7089,17813.0,96.678597,27.317355,141.80,161.644,150,145662,2026-04-16T16:35:00.0000000Z,17813.0,15000.0,0.842082,0.157918


In [47]:
target_names = ["Four-Seamer", "Relay Throw"]
# target_names = ["Four-Seamer", "Rally Cap", "Base Hit", "Relay Throw"]


df = final_df.copy()

df['contest_type'] = None
for name in target_names:
    df.loc[df['contestName'].str.contains(name, case=False, na=False), 'contest_type'] = name

df = df[df['contest_type'].notna()]

pivot = (
    df.pivot_table(
        index='draftGroupId',
        columns='contest_type',
        values=['ev_ratio', 'entryFee', 'average_score', 'p95_score', 'p99_score'],
        aggfunc={
            'ev_ratio': 'mean',
            'entryFee': 'mean',
            'average_score': 'mean',
            'p95_score': 'mean',
            'p99_score': 'mean',
        }
    )
)

# Comparisons
pivot = pivot[['ev_ratio', 'entryFee', 'average_score', 'p95_score', 'p99_score']].dropna(subset=[
    ('average_score', target_names[0]),
    ('average_score', target_names[1])
])

print(pivot.shape)
pivot.describe()

(658, 10)


ev_ratio                entryFee             average_score  \
contest_type Four-Seamer Relay Throw Four-Seamer Relay Throw   Four-Seamer   
count         658.000000  658.000000       658.0       658.0    658.000000   
mean            0.854907    0.863948         4.0        15.0     93.220797   
std             0.035969    0.027396         0.0         0.0     13.787476   
min             0.841043    0.850015         4.0        15.0     54.631502   
25%             0.842318    0.851426         4.0        15.0     83.740529   
50%             0.845109    0.853971         4.0        15.0     92.807605   
75%             0.850340    0.859763         4.0        15.0    102.903934   
max             1.213592    1.079914         4.0        15.0    143.041089   

                           p95_score               p99_score              
contest_type Relay Throw Four-Seamer Relay Throw Four-Seamer Relay Throw  
count         658.000000  658.000000  658.000000  658.000000  658.000000  
mean           92.849164  134.051185  134.445809  149.082352  150.071597  
std            13.200377   19.765274   19.084505   22.235400   21.670768  
min            55.331798   85.660001   87.750000   97.060000   98.350000  
25%            83.712277  119.420625  121.302501  132.491250  134.449245  
50%            92.404464  133.663752  134.317498  149.370000  149.659750  
75%           102.016407  146.398125  146.537500  163.779498  163.709625  
max           147.466417  206.100000  206.100000  220.109990  220.080500

### Analysis
Hypothesis 1: EV increases as entryFee increases
- Findings: Relay Throw ($15) has an EV of 0.86 compared to Four-Seamer's 0.85.
- Conclusion: EV is about the same.

Hypothesis 2: Average competitor performance increases as entryFee increases
- Findings: Relay Throw ($15) contests have an average score of 92.8 points compared to Four-Seamer's 93.2.
- Conclusion: Average score is about the same, with the cheaper contest here actually having a higher average score.

Hypothesis 3: High-end competitor performance increases as entryFee increases
- Findings: Relay Throw ($15) contests have only slightly higher 95th and 99th percentile scores.
- Conclusion: High-end competitor performance is about the same as entryFee increases.

### Take-Aways
Fields aren't terribly different across entryFee. Enter whatever you want. Doesn't matter. 

### Unanswered Questions
- Do more expensive contests distribute payouts differently?
- Would I actually make more or less money in a different contest type?